In [0]:
# ============================================================
# 00.1 BIBLIOTECAS DO FRAMEWORK
# ============================================================
# O QUE FAZ: centraliza as dependências dos Profiles.
# COMO FAZ: importa funções e tipos nativos do PySpark.
# POR QUE É IMPORTANTE: mantém os imports existentes disponíveis.
# PERGUNTA RESPONDIDA: "Quais dependências são compartilhadas?"

from itertools import combinations
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import DataFrame, Window, Column
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.types import (
    NumericType, StringType, DateType, TimestampType,
    BooleanType, MapType, StructType, FloatType,
    DoubleType, ArrayType
)


# Dependências compartilhadas — Machine Learning distribuído (módulo 16).
# São componentes nativos do PySpark ML; não exigem instalação de scikit-learn.
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.functions import vector_to_array

# Dependências compartilhadas — RAG e recuperação lexical (módulo 20).
import json
import re
import math
import hashlib
from urllib.parse import urlparse

In [0]:
# ============================================================
# 00.2 UTILITÁRIOS COMPARTILHADOS
# ============================================================
# O QUE FAZ: padroniza referências e validações reutilizáveis.
# COMO FAZ: utiliza identificadores Spark escapados.
# POR QUE É IMPORTANTE: evita interpretações erradas de nomes especiais.
# PERGUNTA RESPONDIDA: "Como referenciar colunas com segurança?"

def dq_col(nome):
    """Referência segura a um nome literal de coluna, incluindo pontos e crases."""
    if not isinstance(nome, str) or not nome:
        raise ValueError("O nome da coluna deve ser uma string não vazia.")
    return F.col("`" + nome.replace("`", "``") + "`")


def dq_is_atomic(tipo):
    """Política conservadora: exclui arrays, maps e structs."""
    return not isinstance(tipo, (ArrayType, MapType, StructType))


def dq_safe_percentage(numerator, denominator):
    """Percentual Spark; recebe Column ou valor literal; zero retorna NULL."""
    num = numerator if isinstance(numerator, Column) else F.lit(numerator)
    den = denominator if isinstance(denominator, Column) else F.lit(denominator)
    return F.when(den.isNotNull() & (den != 0), num.cast("double") / den.cast("double") * 100.0).otherwise(F.lit(None).cast("double"))


In [0]:
# ============================================================
# FUNÇÃO - COMPRIMENTO DO VALOR
# ============================================================

# O QUE FAZ:
# Calcula o comprimento dos valores utilizados na análise de padrões.

# COMO FAZ:
# Converte o valor para string, remove espaços externos e calcula a quantidade de caracteres.

# POR QUE É IMPORTANTE:
# O comprimento é uma das primeiras características estruturais utilizadas para identificar padrões e possíveis desvios.

# PERGUNTA RESPONDIDA:
# "Qual é o comprimento dos valores presentes na coluna?"

def add_pattern_length(column):
    valor = normalize_pattern_value(column)

    return F.length(valor)

In [0]:
# ============================================================
# 00.3 FUNÇÕES GENÉRICAS — SCHEMA PROFILE
# ============================================================
# O QUE FAZ: define elegibilidade e indicadores de ausência.
# COMO FAZ: utiliza tipos Spark e expressões Column.
# POR QUE É IMPORTANTE: compartilha políticas sem regras de domínio.
# PERGUNTA RESPONDIDA: "Quais análises são aplicáveis ao tipo?"

def dq_schema_supports_cardinality(data_type):
    """Elegibilidade conservadora para contagem de distintos."""
    return dq_is_atomic(data_type)


def dq_schema_is_nan(column, data_type):
    """Detecta NaN somente em Float e Double."""
    return F.isnan(column) if isinstance(data_type, (FloatType, DoubleType)) else F.lit(False)


def dq_schema_is_empty(column, data_type):
    """Detecta string vazia ou composta apenas por espaços."""
    return (column.isNotNull() & (F.trim(column) == "")) if isinstance(data_type, StringType) else F.lit(False)

In [0]:
# ============================================================
# 00.4 FUNÇÕES GENÉRICAS — PATTERN: EXPRESSÕES
# ============================================================
# O QUE FAZ: normaliza e classifica estruturas textuais.
# COMO FAZ: utiliza funções Spark, sem UDFs.
# POR QUE É IMPORTANTE: preserva os contratos do Pattern Profile.
# PERGUNTA RESPONDIDA: "Qual é a estrutura textual do valor?"

def normalize_pattern_value(column):
    return F.trim(column.cast("string"))


def add_pattern_length(column):
    return F.length(normalize_pattern_value(column))


def classify_character_composition(column):
    valor = normalize_pattern_value(column)
    return (
        F.when(valor.isNull() | (valor == ""), "VAZIO")
         .when(valor.rlike(r"^[0-9]+$"), "NUMÉRICO")
         .when(valor.rlike(r"^[A-Za-zÀ-ÿ]+$"), "ALFABÉTICO")
         .when(valor.rlike(r"^[A-Za-zÀ-ÿ0-9]+$") & valor.rlike(r"[A-Za-zÀ-ÿ]") & valor.rlike(r"[0-9]"), "ALFANUMÉRICO")
         .when(valor.rlike(r"^[^A-Za-zÀ-ÿ0-9]+$"), "SÍMBOLOS")
         .otherwise("MISTO")
    )


def generate_pattern_structure(column):
    """L=letra, N=número, S=espaço, X=outros; preserva contrato original."""
    valor = normalize_pattern_value(column)
    return (
        F.when(valor.isNull() | (valor == ""), "VAZIO")
         .otherwise(
             F.regexp_replace(
                 F.regexp_replace(
                     F.regexp_replace(
                         F.regexp_replace(valor, r"[A-Za-zÀ-ÿ]", "L"),
                         r"[0-9]", "N"
                     ), r"\s", "S"
                 ), r"[^LNS]", "X"
             )
         )
    )


In [0]:
# ============================================================
# 00.5 PATTERN — RESUMO DE ESTRUTURAS
# ============================================================
# O QUE FAZ: consolida frequência de estruturas por coluna.
# COMO FAZ: groupBy, unionByName e janela Spark.
# POR QUE É IMPORTANTE: produz ranking de padrões sem pandas.
# PERGUNTA RESPONDIDA: "Quais estruturas predominam?"

def generate_pattern_summary(df, columns, total_records):
    resultados = []
    for coluna in columns:
        estrutura_coluna = (
            df.select(F.lit(coluna).alias("coluna"), generate_pattern_structure(dq_col(coluna)).alias("estrutura"))
              .groupBy("coluna", "estrutura")
              .agg(F.count("*").alias("frequencia"))
              .withColumn("pct_estrutura", dq_safe_percentage(F.col("frequencia"), F.lit(total_records)))
        )
        resultados.append(estrutura_coluna)

    if not resultados:
        vazio_schema = T.StructType([
            T.StructField("coluna", T.StringType(), True),
            T.StructField("estrutura", T.StringType(), True),
            T.StructField("frequencia", T.LongType(), True),
            T.StructField("pct_estrutura", T.DoubleType(), True),
            T.StructField("ranking", T.IntegerType(), True)
        ])
        return df.sparkSession.createDataFrame([], vazio_schema)

    resultado = resultados[0]
    for item in resultados[1:]:
        resultado = resultado.unionByName(item)
    janela = Window.partitionBy("coluna").orderBy(F.desc("frequencia"), F.asc("estrutura"))
    return resultado.withColumn("ranking", F.row_number().over(janela))

In [0]:
# ============================================================
# 00.6 PATTERN — COMPOSIÇÃO DE CARACTERES
# ============================================================
# O QUE FAZ: calcula distribuição das composições por coluna.
# COMO FAZ: groupBy e agregações Spark.
# POR QUE É IMPORTANTE: identifica mudanças estruturais.
# PERGUNTA RESPONDIDA: "Qual composição predomina?"

def generate_character_composition(df, columns, total_records):
    resultados = []
    for coluna in columns:
        composicao = (
            df.select(F.lit(coluna).alias("coluna"), classify_character_composition(dq_col(coluna)).alias("composicao"))
              .groupBy("coluna", "composicao")
              .agg(F.count("*").alias("frequencia"))
              .withColumn("pct_composicao", dq_safe_percentage(F.col("frequencia"), F.lit(total_records)))
        )
        resultados.append(composicao)

    if not resultados:
        vazio_schema = T.StructType([
            T.StructField("coluna", T.StringType(), True),
            T.StructField("composicao", T.StringType(), True),
            T.StructField("frequencia", T.LongType(), True),
            T.StructField("pct_composicao", T.DoubleType(), True)
        ])
        return df.sparkSession.createDataFrame([], vazio_schema)

    resultado = resultados[0]
    for item in resultados[1:]:
        resultado = resultado.unionByName(item)
    return resultado

In [0]:
# ============================================================
# 00.7 FUNÇÕES GENÉRICAS — DUPLICITY
# ============================================================
# O QUE FAZ: avalia elegibilidade e grupos duplicados.
# COMO FAZ: aplica filtros e agregações Spark.
# POR QUE É IMPORTANTE: distingue grupos e excedentes.
# PERGUNTA RESPONDIDA: "Como quantificar duplicidade?"

def dq_valid_value(coluna, tipo, ignorar_texto_vazio=True):
    c = dq_col(coluna)
    valido = c.isNotNull()
    if isinstance(tipo, (FloatType, DoubleType)):
        valido = valido & (~F.isnan(c))
    if ignorar_texto_vazio and isinstance(tipo, StringType):
        valido = valido & (F.length(F.trim(c)) > 0)
    return valido


def dq_duplicate_group_metrics(grupos_df, coluna_frequencia="ocorrencias"):
    f = dq_col(coluna_frequencia)
    return grupos_df.agg(
        F.count(F.lit(1)).alias("grupos_distintos"),
        F.sum(F.when(f > 1, 1).otherwise(0)).alias("grupos_duplicados"),
        F.sum(F.when(f > 1, f).otherwise(0)).alias("registros_em_grupos_duplicados"),
        F.sum(f - 1).alias("linhas_duplicadas_excedentes"),
        F.max(f).alias("maior_grupo")
    )

In [0]:
# ============================================================
# 00.8 FUNÇÕES GENÉRICAS — CONSISTENCY
# ============================================================
# O QUE FAZ: agrega conflitos de dependência funcional.
# COMO FAZ: calcula contagens por grupos previamente agregados.
# POR QUE É IMPORTANTE: centraliza métricas relacionais.
# PERGUNTA RESPONDIDA: "Quantos determinantes são conflitantes?"

def dq_consistency_fd_aggregate(grupos_df):
    conflito = F.col("valores_dependentes_distintos") > 1
    return grupos_df.agg(
        F.sum("registros").alias("registros_validos"),
        F.count(F.lit(1)).alias("determinantes_distintos"),
        F.sum(F.when(conflito, 1).otherwise(0)).alias("determinantes_conflitantes"),
        F.sum(F.when(conflito, F.col("registros")).otherwise(0)).alias("registros_em_grupos_conflitantes")
    )

In [0]:
# ============================================================
# 00.9 FUNÇÕES GENÉRICAS — OUTLIER
# ============================================================
# O QUE FAZ: valida valores e calcula limites estatísticos.
# COMO FAZ: expressões Spark e aritmética de escalares.
# POR QUE É IMPORTANTE: padroniza IQR e Modified Z-Score.
# PERGUNTA RESPONDIDA: "Quais valores são elegíveis e quais limites usar?"

def dq_outlier_col(nome):
    return dq_col(nome)


def dq_outlier_valid(coluna):
    x = coluna.cast("double")
    return coluna.isNotNull() & (~F.isnan(x)) & (F.abs(x) != F.lit(float("inf")))


def dq_outlier_iqr_bounds(q1, q3, factor=1.5):
    if q1 is None or q3 is None or factor <= 0:
        return None, None, None
    iqr = float(q3) - float(q1)
    if iqr < 0:
        return None, None, None
    return float(q1) - factor * iqr, float(q3) + factor * iqr, iqr


def dq_outlier_modified_z_threshold(mad, threshold=3.5):
    if mad is None or mad <= 0 or threshold <= 0:
        return None
    return threshold * float(mad) / 0.6745

In [0]:
# ============================================================
# 00.10 FUNÇÕES GENÉRICAS — CORRELATION
# ============================================================
# O QUE FAZ: prepara valores e classifica correlações.
# COMO FAZ: filtros Spark e cálculo de magnitude.
# POR QUE É IMPORTANTE: padroniza os resultados analíticos.
# PERGUNTA RESPONDIDA: "Como interpretar coeficientes?"

def dq_corr_col(name):
    return dq_col(name)


def dq_corr_valid(name):
    x = dq_corr_col(name).cast("double")
    return x.isNotNull() & (~F.isnan(x)) & (F.abs(x) != F.lit(float("inf")))


def dq_corr_classify(value, weak=0.30, strong=0.80):
    if value is None:
        return "NAO_APLICAVEL", "INDEFINIDA"
    magnitude = abs(float(value))
    intensity = "FORTE" if magnitude >= strong else ("MODERADA" if magnitude >= weak else "FRACA")
    direction = "POSITIVA" if value > 0 else ("NEGATIVA" if value < 0 else "NULA")
    return intensity, direction


def dq_corr_safe_coefficient(value):
    if value is None or not (-float("inf") < float(value) < float("inf")):
        return None
    return max(-1.0, min(1.0, float(value)))

In [0]:
# ============================================================
# 00.11 FUNÇÕES GENÉRICAS — DATA QUALITY SCORE
# ============================================================
# O QUE FAZ: valida percentuais e consolida métricas elegíveis.
# COMO FAZ: agregações Spark com média ponderada.
# POR QUE É IMPORTANTE: preserva rastreabilidade e cobertura.
# PERGUNTA RESPONDIDA: "Como consolidar métricas sem inventar qualidade?"

def dq_score_col(nome):
    return dq_col(nome)


def dq_score_pct(valor):
    x = valor.cast("double")
    valido = x.isNotNull() & (~F.isnan(x)) & (F.abs(x) != F.lit(float("inf")))
    return F.when(valido & (x >= 0) & (x <= 100), x)


def dq_score_metric_schema():
    return T.StructType([
        T.StructField("dimensao", T.StringType(), False),
        T.StructField("coluna", T.StringType(), True),
        T.StructField("metrica", T.StringType(), False),
        T.StructField("valor_observado", T.DoubleType(), True),
        T.StructField("score", T.DoubleType(), True),
        T.StructField("peso", T.DoubleType(), False),
        T.StructField("elegivel", T.BooleanType(), False),
        T.StructField("natureza", T.StringType(), False),
        T.StructField("origem", T.StringType(), False),
        T.StructField("justificativa", T.StringType(), False)
    ])


def dq_score_aggregate(metricas_df, agrupamentos):
    if isinstance(agrupamentos, str):
        agrupamentos = [agrupamentos]
    valido = F.col("elegivel") & F.col("score").isNotNull() & (F.col("peso") > 0)
    base = (
        metricas_df
        .withColumn("_peso_valido", F.when(valido, F.col("peso")).otherwise(F.lit(0.0)))
        .withColumn("_pontos", F.when(valido, F.col("peso") * F.col("score")).otherwise(F.lit(0.0)))
    )
    return (
        base.groupBy(*agrupamentos)
        .agg(
            F.count(F.lit(1)).alias("metricas_catalogadas"),
            F.sum(F.when(valido, 1).otherwise(0)).cast("long").alias("metricas_pontuadas"),
            F.sum("_peso_valido").alias("peso_elegivel"),
            F.sum("_pontos").alias("pontos_ponderados")
        )
        .withColumn("score", F.when(F.col("peso_elegivel") > 0, F.round(F.col("pontos_ponderados") / F.col("peso_elegivel"), 2)))
        .withColumn("cobertura_pct", F.when(F.col("metricas_catalogadas") > 0, F.round(F.col("metricas_pontuadas") * 100.0 / F.col("metricas_catalogadas"), 2)))
    )

In [0]:
# ============================================================
# 00.12 CONFIGURAÇÕES — ORCHESTRATOR
# ============================================================
# O QUE FAZ: centraliza parâmetros de execução.
# COMO FAZ: define configurações uma única vez.
# POR QUE É IMPORTANTE: evita redefinir o run_id e outras opções.
# PERGUNTA RESPONDIDA: "Como configurar a orquestração?"

ORCHESTRATOR_SELECTED = [
    "schema", "distribution", "pattern", "duplicity",
    "consistency", "outlier", "correlation", "score"
]
ORCHESTRATOR_TIMEOUT_SECONDS = 3600
ORCHESTRATOR_STOP_ON_ERROR = True
ORCHESTRATOR_DISPLAY_RESULTS = True
ORCHESTRATOR_NOTEBOOK_ARGUMENTS = {}
ORCHESTRATOR_RUN_ID = str(uuid4())

In [0]:
# ============================================================
# 00.13 FUNÇÕES GENÉRICAS — ORCHESTRATOR
# ============================================================
# O QUE FAZ: valida, planeja e executa os notebooks.
# COMO FAZ: ordenação topológica e dbutils.notebook.run.
# POR QUE É IMPORTANTE: rastreia sucessos, erros e bloqueios.
# PERGUNTA RESPONDIDA: "Como executar Profiles em ordem segura?"

def dq_orchestrator_schema():
    return T.StructType([
        T.StructField("run_id", T.StringType(), False),
        T.StructField("profile", T.StringType(), False),
        T.StructField("status", T.StringType(), False),
        T.StructField("inicio", T.StringType(), False),
        T.StructField("fim", T.StringType(), False),
        T.StructField("duracao_segundos", T.DoubleType(), False),
        T.StructField("mensagem", T.StringType(), True)
    ])


def dq_orchestrator_validate_catalog(catalog):
    if not isinstance(catalog, dict):
        raise TypeError("ORCHESTRATOR_CATALOG deve ser um dicionário.")
    for nome, config in catalog.items():
        if not isinstance(config, dict):
            raise TypeError(f"Configuração inválida para {nome}.")
        if not isinstance(config.get("path"), str):
            raise ValueError(f"Profile {nome} sem path válido.")
        dependencies = config.get("dependencies", [])
        if not isinstance(dependencies, (list, tuple)):
            raise TypeError(f"Dependências inválidas para {nome}.")
        for dependency in dependencies:
            if dependency not in catalog:
                raise ValueError(f"Dependência desconhecida: {dependency}")
    return True


def dq_orchestrator_plan(selected, catalog):
    dq_orchestrator_validate_catalog(catalog)
    if isinstance(selected, str):
        selected = [selected]
    plan, visited, visiting = [], set(), set()

    def visit(profile):
        if profile not in catalog:
            raise ValueError(f"Profile não encontrado: {profile}")
        if profile in visiting:
            raise ValueError(f"Dependência circular: {profile}")
        if profile in visited:
            return
        visiting.add(profile)
        for dependency in catalog[profile].get("dependencies", []):
            visit(dependency)
        visiting.remove(profile)
        visited.add(profile)
        plan.append(profile)

    for profile in selected:
        visit(profile)
    return plan


def dq_orchestrator_execute(dbutils, spark, catalog, selected, run_id,
                            timeout_seconds=3600, arguments=None, stop_on_error=True):
    plan = dq_orchestrator_plan(selected, catalog)
    arguments = arguments or {}
    results, failed = [], set()

    for profile in plan:
        config = catalog[profile]
        dependencies = config.get("dependencies", [])
        if any(dep in failed for dep in dependencies):
            failed.add(profile)
            timestamp = datetime.now(timezone.utc).isoformat()
            results.append((str(run_id), profile, "BLOQUEADO", timestamp,
                            timestamp, 0.0, "Dependência anterior falhou."))
            continue

        inicio = datetime.now(timezone.utc)
        status, mensagem = "SUCESSO", None
        try:
            dbutils.notebook.run(config["path"], int(timeout_seconds), dict(arguments))
        except Exception as exc:
            status, mensagem = "ERRO", str(exc)[:2000]
            failed.add(profile)
        fim = datetime.now(timezone.utc)
        results.append((str(run_id), profile, status, inicio.isoformat(),
                        fim.isoformat(), float((fim - inicio).total_seconds()), mensagem))
        if status == "ERRO" and stop_on_error:
            break

    execution_df = spark.createDataFrame(results, schema=dq_orchestrator_schema())
    return execution_df, plan

In [0]:
# ============================================================
# 00.14 FUNÇÕES GENÉRICAS — MACHINE LEARNING
# ============================================================
# O QUE FAZ:
# Centraliza validações e utilitários reutilizáveis para ML distribuído.
# COMO FAZ:
# Valida parâmetros, identifica atributos numéricos e constrói expressões Spark.
# POR QUE É IMPORTANTE:
# Evita duplicação de contratos e dispensa dependências externas no módulo 16.
# PERGUNTA RESPONDIDA:
# "Quais utilitários compartilhados tornam o ML seguro e reutilizável?"

def dq_ml_validate_parameters(k, seed, max_train_rows, quantile, max_features):
    if not isinstance(k, int) or isinstance(k, bool) or k < 2:
        raise ValueError("ML_K deve ser inteiro >= 2.")
    if not isinstance(seed, int) or isinstance(seed, bool):
        raise ValueError("ML_SEED deve ser inteiro.")
    if not isinstance(max_train_rows, int) or isinstance(max_train_rows, bool) or max_train_rows < k:
        raise ValueError("ML_MAX_TRAIN_ROWS deve ser >= ML_K.")
    if not isinstance(quantile, (float, int)) or isinstance(quantile, bool) or not 0 < quantile < 1:
        raise ValueError("ML_ANOMALY_QUANTILE deve estar entre 0 e 1.")
    if not isinstance(max_features, int) or isinstance(max_features, bool) or max_features < 2:
        raise ValueError("ML_MAX_FEATURES deve ser >= 2.")


def dq_ml_numeric_columns(dataframe, excluded=()):
    if not isinstance(dataframe, DataFrame):
        raise TypeError("A entrada precisa ser um DataFrame Spark.")
    ignored = set(excluded or ())
    return [
        field.name for field in dataframe.schema.fields
        if field.name not in ignored
        and isinstance(field.dataType, NumericType)
        and not isinstance(field.dataType, BooleanType)
    ]


def dq_ml_quote_column(name):
    """Escapa identificadores de coluna Spark, inclusive nomes com pontos."""
    return F.col("`" + str(name).replace("`", "``") + "`")

In [0]:
# ============================================================
# 00.15 FUNÇÕES GENÉRICAS — DATA QUALITY RAG
# ============================================================
# O QUE FAZ:
# Centraliza sanitização, tokenização e validação de fontes RAG.
# COMO FAZ:
# Normaliza rótulos e extrai tokens alfanuméricos de textos curtos.
# POR QUE É IMPORTANTE:
# Reutiliza contratos entre módulos e evita bibliotecas extras.
# PERGUNTA RESPONDIDA:
# "Como preparar textos de recuperação de modo previsível?"

def dq_rag_clean_text(value, max_chars=2000):
    if value is None:
        return ""
    return re.sub(r"[\\x00-\\x1f\\x7f]", " ", str(value)).strip()[:max_chars]


def dq_rag_tokens(value):
    text = dq_rag_clean_text(value, 10000).casefold()
    return sorted(set(re.findall(r"(?u)\\b[^\\W_]{2,}\\b", text)))


def dq_rag_validate_doc_ids(records):
    ids = [r["doc_id"] for r in records]
    if any(not isinstance(v, str) or not v.strip() for v in ids):
        raise ValueError("Todos os documentos precisam de doc_id não vazio.")
    if len(ids) != len(set(ids)):
        raise ValueError("Documentos com doc_id duplicado.")

In [0]:
# ============================================================
# 00.16 TESTES OPCIONAIS - CONTRATOS COMPARTILHADOS
# ============================================================
# O QUE FAZ: verifica nomes e regras básicas sem ler a fonte.
# COMO FAZ: asserts Python e expressões PySpark sintéticas.
# POR QUE É IMPORTANTE: identifica regressões após refatoração.
# PERGUNTA RESPONDIDA: "As funções essenciais continuam disponíveis?"

FRAMEWORK_CONFIG_RUN_TESTS = False

if FRAMEWORK_CONFIG_RUN_TESTS:
    assert dq_is_atomic(T.StringType())
    assert not dq_is_atomic(T.ArrayType(T.StringType()))
    assert dq_schema_supports_cardinality(T.IntegerType())
    assert not dq_schema_supports_cardinality(T.MapType(T.StringType(), T.StringType()))
    assert dq_outlier_iqr_bounds(1, 3) == (-2.0, 6.0, 2.0)
    assert dq_corr_safe_coefficient(1.00000001) == 1.0
    assert dq_orchestrator_plan(["b"], {
        "a": {"path": "./a", "dependencies": []},
        "b": {"path": "./b", "dependencies": ["a"]}
    }) == ["a", "b"]
    print("[OK] Testes básicos do Framework Config aprovados.")
else:
    print("Testes do Framework Config desabilitados.")